In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [3]:
# Datasets and DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms  # transform is the pytorch library that helps in to do transformation on image.

# image => scale (0,1) => normalize (-1,1)

transform = transforms.Compose([
    transforms.ToTensor(), # convert the data into tensor and scales the data
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) # 1st bracket values are std dev. value and 2nd bracket values are mean values.
])
# now we have created a transformal that will be applied to every single image and perform 2 operations.

trainset = CIFAR10(root = "./data", train = True, download = True, transform = transform)

# root tells where the data is 
# train tells that we are selecting the training dataset as CIFAR10 has data divided into train and test data
# download tells that download the data, if not downloaded.
# transform tells that what type of transformation we are going to do on our images.

testset = CIFAR10(root = "./data", train = False, download = True, transform = transform)

In [4]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [5]:
trainloader = DataLoader(trainset, batch_size = 64, shuffle = True)
testloader = DataLoader(testset, batch_size = 64)

# Building CNN

In [15]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.layers = nn.Sequential(

            # 1st convo layer ( Convolution + ReLU + Pooling layer)
            
            nn.Conv2d(3, 32, kernel_size = 3, padding = 1), # 3 => no of channel, 32 => no. of kernel we want to apply on image.
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 2 => kernel dimensions (2x2), 2 => stride value(no. of step to take on filter movement)

                
            # 2nd convo layer ( Convolution + ReLU + Pooling layer)
            
            nn.Conv2d(32, 64, kernel_size = 3, padding = 1), # 32 (output of 1st convo layer) => no of channel, 64 => no. of kernel we want to apply on image.
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 2 => kernel dimensions (2x2), 2 => stride value(no. of step to take on filter movement)
                
            # 3rd convo layer ( Convolution + ReLU + Pooling layer)
            
            nn.Conv2d(64, 128, kernel_size = 3, padding = 1), # 64 (output of 2nd convo layer) => no of channel, 128 => no. of kernel we want to apply on image.
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # 2 => kernel dimensions (2x2), 2 => stride value(no. of step to take on filter movement)
        )

        self.fc_layers = nn.Sequential(

            # hidden layer
            nn.Linear(4*4*128, 256), # 256 => randomly chosen
            nn.ReLU(),

            # Output Layer
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.layers(x)
        x = x.view(x.size(0), -1)  # flatten
        x = self.fc_layers(x)

        return x

In [16]:
model = CNN()

optimizer = optim.Adam(model.parameters())

criterion = nn.CrossEntropyLoss()

# Training of CNN

In [17]:
epochs = 11

for epoch in range(epochs):

    training_loss = 0.0

    for images, labels in trainloader:

        optimizer.zero_grad()
        
        output = model.forward(images)   # forward propagation
        loss = criterion(output, labels) # loss calculate 
        loss.backward()                 # backward propagation
        optimizer.step()                 #  updating parameters

        training_loss += loss.item()

    model.eval()

    with torch.no_grad():
            val_loss = 0.0

            for images,labels in testloader:
                outputs = model.forward(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                
        
    print(f"for {epoch + 1} epoch with training loss = {training_loss/len(trainloader)} and validation loss = {val_loss/len(testloader)}")

for 1 epoch with training loss = 1.3680622613491 and validation loss = 1.0951451108713819
for 2 epoch with training loss = 0.9278016248170067 and validation loss = 0.8523511909375525
for 3 epoch with training loss = 0.7365512929837722 and validation loss = 0.7854980742855436
for 4 epoch with training loss = 0.6085441281728428 and validation loss = 0.7912572553962659
for 5 epoch with training loss = 0.5049757704786633 and validation loss = 0.7728043809817855
for 6 epoch with training loss = 0.41108651998479045 and validation loss = 0.7436753173542631
for 7 epoch with training loss = 0.3237454209028912 and validation loss = 0.8536839103622801
for 8 epoch with training loss = 0.2599732549789617 and validation loss = 0.8504273154932982
for 9 epoch with training loss = 0.19723390972198884 and validation loss = 0.9117685315335632
for 10 epoch with training loss = 0.15863620115163 and validation loss = 1.0325919917434643
for 11 epoch with training loss = 0.12415568807336223 and validation los

# Evaluation 

In [18]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        
        outputs = model.forward(images)
        _,predicted_labels = torch.max(outputs,1)

        correct_labels += (predicted_labels == labels).sum().item()
        total_labels += labels.size(0)

print(f"Total values are {total_labels} and {correct_labels} are correct. So accuracy is {(correct_labels/total_labels)*100}")

Total values are 10000 and 7610 are correct. So accuracy is 76.1
